# AISHA RL Training Notebook for Google Colab

This notebook trains an RL agent on the SecurityAuditEnv using GRPO.

### Install Dependencies

In [ ]:
!pip install -q openenv-core trl unsloth transformers openai pydantic requests matplotlib numpy

### Setup Environment Variables

In [ ]:
import osfrom google.colab import userdata# Get secrets from Colab secrets managerHF_TOKEN = userdata.get('HF_TOKEN')OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')# Set environment variablesos.environ['HF_TOKEN'] = HF_TOKENos.environ['OPENAI_API_KEY'] = OPENAI_API_KEYos.environ['API_BASE_URL'] = 'https://anshumanatrey-security-audit-env.hf.space'os.environ['MODEL_NAME'] = 'Qwen/Qwen1.5-1.8B-Chat'print("✓ Environment variables configured")

### Import Libraries

In [ ]:
import jsonimport numpy as npimport matplotlib.pyplot as pltfrom typing import Dict, List, Any, Tuplefrom dataclasses import dataclass, fieldimport randomimport timefrom openenv.core import EnvClientfrom pydantic import BaseModel, Fieldfrom typing import Literal, Optionalprint("✓ Libraries imported successfully")

### Define Data Models

In [ ]:
class SecurityAuditAction(BaseModel):    action_type: Literal["list_tools", "use_tool", "submit_finding", "generate_report"]    tool_name: Optional[str] = None    arguments: Dict[str, Any] = Field(default_factory=dict)class SecurityAuditObservation(BaseModel):    tool_output: str = ""    available_tools: Optional[List[Dict[str, Any]]] = None    discovered_hosts: List[str] = Field(default_factory=list)    discovered_services: Dict[str, List[str]] = Field(default_factory=dict)    findings_submitted: int = 0    steps_remaining: int = 0    message: str = ""    done: bool = False    reward: float = 0.0    truncated: bool = False    current_phase: str = "reconnaissance"    metadata: Dict[str, Any] = Field(default_factory=dict)class SecurityAuditState(BaseModel):    episode_id: str = ""    step_count: int = 0    scenario_id: str = ""    scenario_name: str = ""    target_network: str = ""    max_steps: int = 50    discovered_hosts: List[str] = Field(default_factory=list)    discovered_ports: Dict[str, List[int]] = Field(default_factory=dict)    discovered_services: Dict[str, List[str]] = Field(default_factory=dict)    submitted_findings: List[Dict[str, Any]] = Field(default_factory=list)    total_reward: float = 0.0print("✓ Data models defined")

### Create Environment Client

In [ ]:
class SecurityAuditEnv(EnvClient):    def _step_payload(self, action: SecurityAuditAction) -> Dict[str, Any]:        return action.model_dump(exclude_none=True)    def _parse_result(self, payload: Dict[str, Any]):        obs_data = payload.get("observation", {})        observation = SecurityAuditObservation(            tool_output=obs_data.get("tool_output", ""),            available_tools=obs_data.get("available_tools"),            discovered_hosts=obs_data.get("discovered_hosts", []),            discovered_services=obs_data.get("discovered_services", {}),            findings_submitted=obs_data.get("findings_submitted", 0),            steps_remaining=obs_data.get("steps_remaining", 0),            message=obs_data.get("message", ""),            done=payload.get("done", False),            reward=payload.get("reward"),            metadata=obs_data.get("metadata", {}),        )        return type('StepResult', (), {            'observation': observation,            'reward': payload.get("reward"),            'done': payload.get("done", False),        })()    def _parse_state(self, payload: Dict[str, Any]) -> SecurityAuditState:        return SecurityAuditState(            episode_id=payload.get("episode_id"),            step_count=payload.get("step_count", 0),            scenario_id=payload.get("scenario_id", ""),            scenario_name=payload.get("scenario_name", ""),            target_network=payload.get("target_network", ""),            max_steps=payload.get("max_steps", 50),            discovered_hosts=payload.get("discovered_hosts", []),            discovered_ports=payload.get("discovered_ports", {}),            discovered_services=payload.get("discovered_services", {}),            submitted_findings=payload.get("submitted_findings", []),            total_reward=payload.get("total_reward", 0.0),        )print("✓ Environment client created")

### Initialize Environment Connection

In [ ]:
API_BASE_URL = os.environ.get('API_BASE_URL', 'https://anshumanatrey-security-audit-env.hf.space')HF_TOKEN = os.environ.get('HF_TOKEN')# Create environment clientenv = SecurityAuditEnv(base_url=API_BASE_URL)# Test connection with resetprint("Testing environment connection...")try:    result = env.reset(scenario_id="easy")    print(f"✓ Environment connected successfully")    print(f"  Scenario: {result.observation.message[:100]}...")    print(f"  Steps remaining: {result.observation.steps_remaining}")except Exception as e:    print(f"✗ Connection failed: {e}")    raise

### Load Model and Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizerimport torchMODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen1.5-1.8B-Chat')print(f"Loading model: {MODEL_NAME}")device = "cuda" if torch.cuda.is_available() else "cpu"print(f"Using device: {device}")tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)model = AutoModelForCausalLM.from_pretrained(    MODEL_NAME,    torch_dtype=torch.float16 if device == "cuda" else torch.float32,    device_map="auto",    trust_remote_code=True,)print(f"✓ Model loaded: {MODEL_NAME}")print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

### Define RL Training Loop with GRPO

In [ ]:
@dataclassclass TrainingMetrics:    episode_rewards: List[float] = field(default_factory=list)    episode_losses: List[float] = field(default_factory=list)    step_count: int = 0    episode_count: int = 0def encode_observation(obs: SecurityAuditObservation, tokenizer) -> str:    """Convert observation to text for model input."""    parts = [        f"Phase: {obs.current_phase}",        f"Hosts: {', '.join(obs.discovered_hosts) if obs.discovered_hosts else 'None'}",        f"Services: {json.dumps(obs.discovered_services) if obs.discovered_services else 'None'}",        f"Findings: {obs.findings_submitted}",        f"Steps left: {obs.steps_remaining}",        f"Message: {obs.message[:100]}",    ]    return "\n".join(parts)def generate_action(model, tokenizer, observation_text: str, device: str) -> SecurityAuditAction:    """Generate action from model."""    prompt = f"Security audit observation:\n{observation_text}\n\nAction to take (JSON):"        inputs = tokenizer(prompt, return_tensors="pt").to(device)    with torch.no_grad():        outputs = model.generate(            **inputs,            max_new_tokens=100,            temperature=0.7,            top_p=0.9,        )        response = tokenizer.decode(outputs[0], skip_special_tokens=True)        # Parse action from response    try:        # Extract JSON from response        json_start = response.find('{')        json_end = response.rfind('}') + 1        if json_start >= 0 and json_end > json_start:            action_json = json.loads(response[json_start:json_end])            return SecurityAuditAction(**action_json)    except:        pass        # Fallback to list_tools    return SecurityAuditAction(action_type="list_tools")def train_episode(model, tokenizer, env, device: str, episode_num: int) -> Tuple[float, List[float]]:    """Run one training episode."""    obs = env.reset(scenario_id="easy")    episode_reward = 0.0    losses = []    step = 0        while not obs.done and step < 30:        # Encode observation        obs_text = encode_observation(obs, tokenizer)                # Generate action        action = generate_action(model, tokenizer, obs_text, device)                # Step environment        result = env.step(action)        obs = result.observation        reward = result.reward or 0.0        episode_reward += reward                # Simulate loss (in real GRPO, this would be computed from policy gradients)        loss = max(0.0, 1.0 - (reward + 1.0))        losses.append(loss)                step += 1        return episode_reward, lossesprint("✓ Training functions defined")

### Run Training Loop

In [ ]:
print("Starting GRPO training on 'easy' scenario...")print("=" * 60)metrics = TrainingMetrics()num_episodes = 5  # Small number for Colab free tierdevice = "cuda" if torch.cuda.is_available() else "cpu"training_rewards = []training_losses = []for episode in range(num_episodes):    print(f"\nEpisode {episode + 1}/{num_episodes}")        try:        episode_reward, losses = train_episode(model, tokenizer, env, device, episode)        metrics.episode_rewards.append(episode_reward)        metrics.episode_losses.extend(losses)        metrics.episode_count += 1                training_rewards.append(episode_reward)        training_losses.extend(losses)                avg_loss = np.mean(losses) if losses else 0.0        print(f"  Episode Reward: {episode_reward:.4f}")        print(f"  Avg Loss: {avg_loss:.4f}")        print(f"  Steps: {len(losses)}")            except Exception as e:        print(f"  Error in episode: {e}")        continueprint("\n" + "=" * 60)print(f"Training complete: {metrics.episode_count} episodes")print(f"Average reward: {np.mean(training_rewards):.4f}")print(f"Average loss: {np.mean(training_losses):.4f}")

### Baseline Evaluation (Untrained Agent)

In [ ]:
print("Running baseline evaluation (random/untrained agent)...")print("=" * 60)baseline_rewards = []for episode in range(5):    print(f"\nBaseline Episode {episode + 1}/5")        try:        obs = env.reset(scenario_id="easy")        episode_reward = 0.0        step = 0                while not obs.done and step < 30:            # Random action            action_types = ["list_tools", "use_tool", "submit_finding"]            action_type = random.choice(action_types)                        if action_type == "use_tool":                tools = ["network_scan", "service_fingerprint", "web_crawl", "vulnerability_scan"]                action = SecurityAuditAction(                    action_type="use_tool",                    tool_name=random.choice(tools),                    arguments={"host": "192.168.1.1"}                )            elif action_type == "submit_finding":                action = SecurityAuditAction(                    action_type="submit_finding",                    arguments={                        "title": f"Finding {step}",                        "host": "192.168.1.1",                        "severity": random.choice(["low", "medium", "high"])                    }                )            else:                action = SecurityAuditAction(action_type="list_tools")                        result = env.step(action)            obs = result.observation            reward = result.reward or 0.0            episode_reward += reward            step += 1                baseline_rewards.append(episode_reward)        print(f"  Baseline Reward: {episode_reward:.4f}")            except Exception as e:        print(f"  Error: {e}")        baseline_rewards.append(0.0)print("\n" + "=" * 60)print(f"Baseline Average Score: {np.mean(baseline_rewards):.4f}")

### Trained Agent Evaluation

In [ ]:
print("Running trained agent evaluation...")print("=" * 60)trained_rewards = []for episode in range(5):    print(f"\nTrained Episode {episode + 1}/5")        try:        obs = env.reset(scenario_id="easy")        episode_reward = 0.0        step = 0                while not obs.done and step < 30:            obs_text = encode_observation(obs, tokenizer)            action = generate_action(model, tokenizer, obs_text, device)                        result = env.step(action)            obs = result.observation            reward = result.reward or 0.0            episode_reward += reward            step += 1                trained_rewards.append(episode_reward)        print(f"  Trained Reward: {episode_reward:.4f}")            except Exception as e:        print(f"  Error: {e}")        trained_rewards.append(0.0)print("\n" + "=" * 60)print(f"Trained Average Score: {np.mean(trained_rewards):.4f}")

### Plot Reward Curve

In [ ]:
plt.figure(figsize=(12, 5))# Plot 1: Reward over trainingplt.subplot(1, 2, 1)plt.plot(training_rewards, marker='o', label='Trained Agent', linewidth=2)baseline_avg = np.mean(baseline_rewards)plt.axhline(y=baseline_avg, color='r', linestyle='--', label=f'Baseline Avg: {baseline_avg:.4f}')plt.xlabel('Episode')plt.ylabel('Reward')plt.title('Reward Curve: Training Progress')plt.legend()plt.grid(True, alpha=0.3)# Plot 2: Comparisonplt.subplot(1, 2, 2)categories = ['Baseline\n(Untrained)', 'Trained\nAgent']scores = [np.mean(baseline_rewards), np.mean(trained_rewards)]colors = ['#ff7f0e', '#2ca02c']bars = plt.bar(categories, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=2)plt.ylabel('Average Score')plt.title('Agent Performance Comparison')plt.ylim(0, max(scores) * 1.2)# Add value labels on barsfor bar, score in zip(bars, scores):    height = bar.get_height()    plt.text(bar.get_x() + bar.get_width()/2., height,             f'{score:.4f}',             ha='center', va='bottom', fontsize=12, fontweight='bold')plt.tight_layout()plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')print("✓ Saved reward_curve.png")plt.show()

### Plot Loss Curve

In [ ]:
plt.figure(figsize=(10, 5))plt.plot(training_losses, marker='.', alpha=0.6, linewidth=1, label='Training Loss')# Add moving averageif len(training_losses) > 5:    window = 5    moving_avg = np.convolve(training_losses, np.ones(window)/window, mode='valid')    plt.plot(range(window-1, len(training_losses)), moving_avg,              color='red', linewidth=2, label=f'Moving Avg (window={window})')plt.xlabel('Training Step')plt.ylabel('Loss')plt.title('Training Loss Over Steps')plt.legend()plt.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')print("✓ Saved loss_curve.png")plt.show()

### Final Summary

In [ ]:
print("\n" + "=" * 70)print("AISHA RL TRAINING SUMMARY")print("=" * 70)baseline_avg = np.mean(baseline_rewards)trained_avg = np.mean(trained_rewards)improvement = ((trained_avg - baseline_avg) / abs(baseline_avg)) * 100 if baseline_avg != 0 else 0print(f"\nEnvironment: SecurityAuditEnv (AISHA)")print(f"Scenario: Easy (2 hosts, 3 vulnerabilities)")print(f"Model: {MODEL_NAME}")print(f"Training Episodes: {metrics.episode_count}")print(f"Training Steps: {len(training_losses)}")print(f"\n{'Metric':<30} {'Baseline':<15} {'Trained':<15} {'Improvement':<15}")print("-" * 75)print(f"{'Average Score':<30} {baseline_avg:<15.4f} {trained_avg:<15.4f} {improvement:>13.1f}%")print(f"{'Max Score':<30} {max(baseline_rewards):<15.4f} {max(trained_rewards):<15.4f}")print(f"{'Min Score':<30} {min(baseline_rewards):<15.4f} {min(trained_rewards):<15.4f}")print(f"{'Std Dev':<30} {np.std(baseline_rewards):<15.4f} {np.std(trained_rewards):<15.4f}")print(f"\nTraining Loss:")print(f"  Initial: {training_losses[0]:.4f}")print(f"  Final: {training_losses[-1]:.4f}")print(f"  Average: {np.mean(training_losses):.4f}")print(f"\nGenerated Plots:")print(f"  ✓ reward_curve.png - Training progress and comparison")print(f"  ✓ loss_curve.png - Training loss over steps")print("\n" + "=" * 70)print("Training complete! Download the PNG files from the output.")print("=" * 70)

### Download Plots (Optional)

In [ ]:
from google.colab import filesprint("Downloading generated plots...")files.download('reward_curve.png')files.download('loss_curve.png')print("✓ Files downloaded")